# 7.3 Finite diffraction beams and spatial overlap

In [7.2](7_2_Interference_of_Diffraction_Orders_from_a_Grating.ipynb), ideal plane-wave orders fill the output half-space. Separating their arrows does not separate their fields. A finite beam has an envelope. Its axis can separate from another beam while diffraction changes its width. Here we examine how those changes affect detector overlap and interference.

:::{note} Scope of the intensity comparison

 Relative $|U|^2$ uses a fixed-medium reference. It does not compare absolute irradiance across media.
:::

## Motivation and prerequisites

Here we assign two analytical Gaussian beams to grating-order directions and follow their overlap. Their widths and amplitudes are inputs, so the calculation describes those chosen beams. Calculating the output field from a finitely illuminated grating is a different problem; the prescribed boundary-field treatment is in [Appendix 2](Appendix_2_Numerical_Diffraction_of_Finite_Grating_Illumination.ipynb), with its numerical limitations documented.

Review [0.2: Gaussian-beam theory](0_2_Solutions_of_the_Helmholtz_Equation.ipynb), [1.2: ideal versus finite beams](1_2_Ideal_Plane_Waves_versus_Finite_Optical_Beams.ipynb), [2.3: allowed orders](2_3_Allowed_Diffraction_Orders.ipynb), and [7.1: coherence and visibility](7_1_Interference_and_Mutual_Coherence_of_Two_Plane_Waves.ipynb). Run the cells in order from a fresh kernel.

## 1. From ideal orders to finite beams

The explorer places the assigned beams and detector in a common geometry.

- **3D view:** green shows the prescribed incident beam; blue and orange show the assigned output beams. Arrows mark propagation axes. Translucent surfaces mark the local $1/e^2$ intensity radius, not beam boundaries. **XY detector:** move the plane and inspect the combined intensity in the adjacent map.
- The square viewing window follows the beam footprints. It is a plotting crop, not a calculated detector or grating size. The dashed line marks $y=0$. Optional xz views show individual intensities, their sum and the signed cross term.
- The incident and output beams are prescribed; this is not a grating-scattering calculation.

Drag the 3D scene to rotate; use the camera toolbar to zoom or pan. Your view is retained when changing experiment settings.

**Try:**

Move the detector from the grating to 80 µm depth. Predict how the envelope peaks will move. Compare the detector-line profile with the overlap curve. Reset, then increase the waist. Does overlap persist farther downstream?

**The Gaussian output beams are prescribed.** We assign their widths and amplitudes independently. The grating equation supplies their central directions, but this model does not calculate the output profiles or diffraction efficiencies of a particular grating.

**Directions and assigned shapes.** The grating equation sets each beam's central direction. At normal incidence, $n\sin\theta_m=m\lambda_0/d$, equivalently $\sin\theta_m=m\lambda/d$ with $\lambda=\lambda_0/n$: the default orders $0$ and $+1$ leave at 0° and about 14.5°. We then assign a Gaussian beam with a chosen waist to each direction.

The wavelength is measured in one homogeneous medium. The grating is at $z=0$, transmission is toward negative $z$, and each assigned beam's waist centre is at the origin. Widths are measured perpendicular to each beam's own axis. Both coefficient magnitudes are one, and their relative phase is stable.

Detailed coordinates and assumptions are available in the optional background.

In [ ]:
from pathlib import Path
import sys
import numpy as np
from IPython.display import display

_support_dir = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within Grating_Physics.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
from wave_optics.gaussian_order_beams import (
    GaussianOrderConfiguration, GaussianOrderExperiment, line_overlap)
from wave_optics.gaussian_notebook_setup import create_gaussian_explorer

In [ ]:
gaussian_orders = create_gaussian_explorer(globals().get('gaussian_orders'))
display(gaussian_orders.ui)

## 2. Separation, spreading and spatial overlap

Move the detector away from the grating. Two changes happen together: the beam centres move apart because the beams travel in different directions, and diffraction broadens each Gaussian beam. The linked profiles below show both effects on the same intensity scale.

The outer profiles are references at depths 0 and 80 µm. The middle profile follows your selected detector. Change the waist and order pair using the controls above: which change helps the beams keep a shared illuminated region farther downstream?

The overlap number summarizes how much the two intensity envelopes coincide along the central detector line. It is close to one for matching envelopes and smaller when they separate; it is undefined if either beam vanishes. The green shading shows the product of the two field magnitudes used to calculate it.

**Less spatial overlap does not mean less mutual coherence.** These fields keep a deterministic phase relationship. Interference becomes weak where one of the fields is negligible, even though their mutual-coherence magnitude remains one wherever both are nonzero. Spreading may partly counter separation, so overlap need not decrease monotonically for every geometry.

The coordinate treatment and the exact overlap formula are optional. The main task is to compare where the light arrives, how wide each beam is, and where both fields remain appreciable.

In [ ]:
display(gaussian_orders.lesson_aids.overlap_view())

**Predict, change, interpret:** compare waists 4, 6 and 12 µm. A wider waist starts with more overlap; a narrower waist also diffracts more strongly. Then select $(-1,+1)$ and compare the increased separation. Finally change only the relative phase: the individual profiles and overlap stay unchanged.

### Small calculation 1 — overlap and phase

This calculation takes a snapshot of the current controls; rerun it after changing them.

In [ ]:
beam_model = gaussian_orders.model
x_um = gaussian_orders.x
sample = beam_model.sample(x_um, np.array([0., gaussian_orders.detector.value, 80.]),
    pair=gaussian_orders.pair.value,
    coefficients=(1., np.exp(1j*np.deg2rad(gaussian_orders.phase.value))))
for depth_um, overlap in zip(sample['depth'], sample['overlap']):
    print(f'Depth {depth_um:g} µm: detector-line overlap = {overlap:.6f}')
print('Phase invariance:', np.max(abs(line_overlap(
    sample['fields'][0], 1j*sample['fields'][1], x_um)-sample['overlap'])))

:::{dropdown} Answer — what changed?

The overlap changes with beam geometry and detector position. Multiplying one field by $i$ changes its phase but leaves the overlap unchanged to rounding accuracy. The number describes the intensity envelopes, not mutual coherence or measured visibility.

:::

## 3. Coherent interference of finite beams

The assigned complex fields have a stable relative phase. With their coefficient phases included,

$$I=|U_m+U_p|^2=I_m+I_p+2\operatorname{Re}(U_m^*U_p).$$

The signed cross-term map shows constructive and destructive contributions. It can be negative, while total intensity remains nonnegative. Where only one beam is appreciable, the cross term is small. The absence of visible fringes there does not establish loss of mutual coherence: there is too little second-field amplitude to produce strong modulation.

### What sets the fringe spacing?

Fringes appear because the relative phase changes as we move across the detector. At a fixed detector plane, write

$$\Delta\phi(x)=\arg\!\left[U_m^*(x,0,z_D)U_p(x,0,z_D)\right],$$

$$I=I_m+I_p+2\sqrt{I_mI_p}\cos\Delta\phi.$$

A $2\pi$ change in relative phase produces one interference cycle. **A faster phase change across the detector gives more closely spaced fringes.**

For two plane waves of the same wavelength, $\Delta\phi(x)=\Delta k_x x+\text{constant}$, with $\Delta k_x=k_{x,p}-k_{x,m}$. Their nominal spacing is

$$\boxed{\Lambda_x
=\frac{2\pi}{|\Delta k_x|}
=\frac{\lambda}{|\sin\theta_p-\sin\theta_m|}
=\frac{d}{|p-m|}.}$$

Angles are signed from $-z$ (the transmitted detector normal) toward positive $x$. The last equality uses the grating equation for two orders of the same incident monochromatic field. Thus a larger order difference or smaller grating pitch gives finer nominal fringes. Wavelength cancels from this final expression for fixed orders; it still affects their directions and Gaussian propagation.

For beams symmetric about the detector normal, with angular separation $\Delta\theta$,

$$\boxed{\Lambda_x=\frac{\lambda}{2\sin(\Delta\theta/2)}.}$$

**Greater angular separation means finer fringes in this symmetric geometry.** More generally, it is the transverse wavevector difference that determines spacing on the detector.

### Why can the Gaussian-beam spacing differ?

Plane waves have a constant phase gradient. A Gaussian beam has curved wavefronts, so the two beams' relative phase gradient can vary across the detector. Here “phase gradient” simply means phase change per unit distance. Where both fields are appreciable, define the local spacing estimate by

$$\boxed{\Lambda_{\mathrm{local}}(x)
=\frac{2\pi}{\left|d\Delta\phi(x)/dx\right|}.}$$

The phase is unwrapped before differentiating. This is the distance for a $2\pi$ phase change if the gradient stays approximately constant locally. It agrees with adjacent intensity-maxima spacing when the gradient and envelopes vary little over a cycle; varying envelopes can shift those maxima.

**Worked default example:** $\lambda=1$ µm, $d=4$ µm and orders $(0,+1)$ give $\theta_0=0^\circ$ and $\theta_1=\arcsin(1/4)\approx14.48^\circ$. Therefore

$$\Lambda_x=\frac{1\ \mu\mathrm{m}}{|\sin14.48^\circ-\sin0^\circ|}
\approx4\ \mu\mathrm{m}.$$

For the actual Gaussian fields with waist 6 µm, the local estimate near strongest overlap is about **4.50 µm at the default depth of 40 µm**. At 80 µm depth it is about 6.00 µm. Curvature effects are therefore not always small; a broad-beam limit approaches the nominal relation.

The two rulers represent different spacing estimates: cyan marks the nominal carrier spacing, while purple marks the local Gaussian estimate. Open **Fringe spacing** to compare them across the detector. Compare $(0,+1)$ with $(-1,+1)$: the nominal spacing changes from 4 to 2 µm at pitch 4 µm. Then change depth or waist to inspect curvature effects. A fixed coefficient phase shifts the fringes without changing either spacing estimate.

### Visibility and phase control

From 7.1, a local phase scan has visibility

$$V(\mathbf r)=\frac{2\sqrt{\bar I_m\bar I_p}}{\bar I_m+\bar I_p}|\gamma_{mp}|.$$

This deterministic model assumes $|\gamma_{mp}|=1$ wherever both fields are nonzero. Unequal local amplitudes can still reduce visibility. The formula describes modulation during a phase scan at a fixed point; fitting neighboring spatial extrema is only equivalent when the envelopes vary slowly over a fringe. Detector pixels and exposure averaging can further change measured contrast and are not simulated here.

**Predict, change, interpret:** hold geometry and detector fixed, then change relative coefficient phase $\beta$. Fringes shift and the signed cross term changes. Individual intensities and spatial overlap stay fixed. A phase change does not redirect an optical axis or reduce mutual-coherence magnitude. With a broad beam, a small central region approaches the two-plane-wave result from 7.2; curvature and envelope variation limit that approximation over larger regions.

At a detector location near one envelope peak but far from the other, pause the phase scan. Predict the largest possible oscillation of intensity there from the two individual curves. The product of their field magnitudes sets the cross-term amplitude. Near an appreciable intersection with comparable intensities, a phase scan can produce deep modulation; in a weak tail, a mathematically defined contrast can be experimentally unhelpful because very little light arrives.

### Small calculation 2 — test coherent addition

In [ ]:
U_m, U_p = sample['fields']
identity_error = np.max(abs(sample['intensity']
    -abs(U_m)**2-abs(U_p)**2-sample['cross']))
single_arm = beam_model.sample(x_um, [gaussian_orders.detector.value],
    pair=gaussian_orders.pair.value, coefficients=(1., 0.))
print('Intensity identity error:', identity_error)
print('Single-arm overlap (undefined):', single_arm['overlap'][0])
print('Single-arm cross term:', np.max(abs(single_arm['cross'])))

## Worked-out examples

These optional scripts show the numerical implementation of the topics in this notebook.
Use the complete [scripts download](Downloads.md) to keep the plotting helper and
`wave_optics` package with the examples. Edit **INPUT PARAMETERS**, then run the file
or execute its `# %%` sections in order. Read **CHECK AND INTERPRET** with the plots.

### 1. From ideal orders to finite beams

**Script:** [7_3_01_rotated_gaussian_implementation.py](Wave_solver_py_scripts/scripts/chapter_7/7_3_01_rotated_gaussian_implementation.py)

Implement a Gaussian beam in its own rotated coordinate frame.

### 2. Separation spreading and spatial overlap

**Script:** [7_3_02_overlap_quadrature.py](Wave_solver_py_scripts/scripts/chapter_7/7_3_02_overlap_quadrature.py)

Integrate detector-line overlap for two spreading order beams.

### 3. Coherent interference of finite beams

**Script:** [7_3_03_local_fringe_phase.py](Wave_solver_py_scripts/scripts/chapter_7/7_3_03_local_fringe_phase.py)

Calculate finite-beam fringe phase and its local spatial frequency.

### 4. Visibility and finite-beam overlap

**Script:** [7_3_04_local_visibility_and_overlap.py](Wave_solver_py_scripts/scripts/chapter_7/7_3_04_local_visibility_and_overlap.py)

Distinguish local fringe visibility from integrated beam overlap.


:::{dropdown} Answer — one arm blocked

The identity holds to rounding error. Blocking either arm removes the cross term and leaves the other beam's intensity. Normalized two-beam overlap is NaN because its denominator vanishes. It would be misleading to interpret that undefined value as a measured coherence or visibility.

:::

## Final checkpoint

1. What determines the order directions, and what determines the assigned beam widths?
2. Why do indefinitely extended plane waves overlap even when their arrows separate?
3. Why can spreading fail to preserve strong overlap?
4. What does the overlap curve measure, and what does it leave out?
5. Can changing $\beta$ restore spatial overlap?
6. What additional question does Appendix 2 address?
7. What sets nominal fringe spacing, and why can the local Gaussian spacing differ?

:::{dropdown} Checkpoint answers

1. The grating equation sets directions; the prescribed waist and Gaussian propagation set widths. Efficiencies are not calculated.
2. An ideal plane wave has no finite transverse envelope or hard boundary.
3. The beams broaden, but their centres can move apart faster than spreading compensates.
4. Normalized magnitude overlap along the central detector line; not full-plane power, mutual coherence, or measured fringe visibility.
5. No. It changes the relative phase and cross term, not the envelopes or directions.
6. It propagates a finite illuminated binary-mask boundary numerically and examines approximate spectral-order isolation, subject to sampling, padding and filtering limitations.
7. For the same-wavelength plane-wave reference, $\Lambda_x=2\pi/|\Delta k_x|=\lambda/|\sin\theta_p-\sin\theta_m|=d/|p-m|$. Curved Gaussian wavefronts change the local relative-phase gradient. The default nominal value is 4 µm, while the local estimate near strongest overlap at depth 40 µm is about 4.50 µm.

:::

**Previous:** [7.2 — ideal diffraction-order interference](7_2_Interference_of_Diffraction_Orders_from_a_Grating.ipynb). **Numerical extension:** [Appendix 2](Appendix_2_Numerical_Diffraction_of_Finite_Grating_Illumination.ipynb). **Next:** [7.4: diffraction kinematics and complex coefficients](7_4_Diffraction_Kinematics_and_Complex_Order_Coefficients.ipynb).



(refs-7-3)=

## Further Reading

(ref-7-3-siegman)=

- A. E. Siegman, *Lasers*. University Science Books, 1986. [Online]. Available: [source](https://mitpress.mit.edu/9780935702118/lasers/).

  **Reading notes:** A. E. Siegman, *Lasers* (University Science Books, 1986), Gaussian-beam propagation.

(ref-7-3-goodman-2017)=

- J. W. Goodman, *Introduction to Fourier Optics*, 4th ed. W. H. Freeman, 2017. [Online]. Available: [source](https://www.macmillanlearning.com/ed/uk/product/Introduction-to-Fourier-Optics-4th-edition/p/1319119166).

  **Reading notes:** J. W. Goodman, *Introduction to Fourier Optics*, 4th ed. (W. H. Freeman, 2017), scalar diffraction and coherent superposition.

(ref-7-3-gaussian)=

- F. Pampaloni and J. Enderlein, “Gaussian, Hermite-Gaussian, and Laguerre-Gaussian beams: A primer,” arXiv:physics/0410021, 2004. [Online]. Available: [preprint](https://arxiv.org/abs/physics/0410021).

  **Reading notes:** **Access:** Free arXiv preprint; not public domain. **Read for:** Read pp. 3–8: plane-wave expansion and fundamental Gaussian mode.

(ref-7-3-gbur-visser)=

- G. Gbur and T. D. Visser, “The Structure of Partially Coherent Fields,” *Progress in Optics*, vol. 55, pp. 285–341, 2010. [Online]. Available: [PDF](https://www.nat.vu.nl/~tvisser/PinO.pdf). doi: 10.1016/B978-0-444-53705-8.00005-9.

  **Reading notes:** **Access:** Free author-hosted review chapter; copyright retained. **Read for:** Read Sections 2–3: correlation, interference and visibility.

## Reference notes

Freely accessible works are not necessarily in the public domain.



## Course connections

[0.2 — Solutions of the Helmholtz equation](0_2_Solutions_of_the_Helmholtz_Equation.ipynb): Gaussian width, curvature and Gouy phase; Gaussian beams are paraxial approximations. [1.2 — Ideal plane waves versus finite optical beams](1_2_Ideal_Plane_Waves_versus_Finite_Optical_Beams.ipynb): ideal extent and experimental envelopes. [7.1 — Interference and mutual coherence](7_1_Interference_and_Mutual_Coherence_of_Two_Plane_Waves.ipynb): local amplitude balance, phase and visibility.


## Optional background

:::{dropdown} Beam directions, local coordinates and Gaussian spreading

The central wavevector follows

$$k_{x,m}=k_{ix}+mG,\qquad G=\frac{2\pi}{d},\qquad k=\frac{2\pi}{\lambda},\qquad
k_{z,m}=-\sqrt{k^2-k_{x,m}^2}.$$

Strictly propagating centres require $|k_{x,m}|<k$. At normal incidence $k_{ix}=0$, and $n\sin\theta_m=m\lambda_0/d$, equivalently $\sin\theta_m=m\lambda/d$ with $\lambda=\lambda_0/n$. The explorer restricts axes to 30° from negative $z$. Separately, its waist controls keep $w_0/\lambda\ge3.33$, within the declared weak-divergence teaching regime.

With waist centre at the origin, define the propagation and transverse unit vectors

$$\hat{\mathbf l}_m=(\sin\theta_m,0,-\cos\theta_m),\qquad
\hat{\mathbf q}_m=(-\cos\theta_m,0,-\sin\theta_m),$$

$$\ell_m=\mathbf r\cdot\hat{\mathbf l}_m,\qquad
q_m=\mathbf r\cdot\hat{\mathbf q}_m,\qquad \rho_m^2=q_m^2+y^2.$$

The shared Gaussian helper is evaluated at $(q_m,y,\ell_m)$. This rotates the field's envelope, curvature and Gouy phase together. The carrier phase gradient is exactly $k\hat{\mathbf l}_m$. Each waist plane is perpendicular to its own axis; the grating plane is not a common waist plane for tilted beams. This is not a finite Gaussian boundary multiplied by a mask.

The local transverse radius and Rayleigh range are

$$w(\ell)=w_0\sqrt{1+(\ell/z_R)^2},\qquad z_R=\frac{\pi w_0^2}{\lambda}.$$

At detector depth $s=-z_D$, each axis crosses at $x=s\tan\theta_m$. Across a horizontal detector, both local longitudinal and transverse coordinates vary. The helper evaluates that oblique slice directly rather than simply translating a fixed Gaussian profile. The xz figures sample the axisymmetric three-dimensional helper at $y=0$; they are not exact two-dimensional slit-scattering solutions. Fixed waist amplitude does not imply fixed total beam power when comparing different waists.

:::

:::{dropdown} Exact definition of the detector-line overlap

$$\mathcal O_{\mathrm{line}}(z_D)=
\frac{\int |U_m(x,0,z_D)||U_p(x,0,z_D)|\,dx}
{\sqrt{\int |U_m|^2dx\int |U_p|^2dx}}.$$

The implementation integrates over $-80\le x\le80$ µm, independently of the narrower fitted display crop. This is a central-line diagnostic, not full XY-plane overlap or optical power. Cauchy–Schwarz bounds it between zero and one for nonzero arms; a zero denominator gives NaN. Scaling either nonzero field or adding a fixed coefficient phase leaves it unchanged. Detector-line sampling and extension were checked for the documented validation cases; those checks do not turn this quantity into coherence or measured visibility.

:::

In [ ]:
display(gaussian_orders.lesson_aids.coordinates_panel())

### Optional calculation — default direction and spreading scales

This independent numerical example uses the default parameters, not the current control settings.

In [ ]:
default_model = GaussianOrderExperiment()
default_c = default_model.configuration
for order in (0, 1):
    direction = default_model.basis(order)[1]
    print(f'Order {order:+d}: k vector = {default_model.k*direction} rad/µm')
print(f'Rayleigh range: {default_c.rayleigh:.3f} µm')
print(f'Far-field divergence: {default_c.wavelength/(np.pi*default_c.waist):.4f} rad')

:::{dropdown} Answer — default values

At $\lambda=1$, $d=4$, $w_0=6$ µm, the order angles are 0° and approximately 14.48°. Their wavevectors are $(0,0,-6.283)$ and $(1.571,0,-6.084)$ rad/µm. The Rayleigh range is about 113.10 µm and the local divergence estimate is 0.0531 rad. These are assigned beam properties, not calculated grating efficiencies.

:::

:::{dropdown} Optional — phase conventions and numerical display checks

The shared helper evaluates

$$U=C\frac{w_0}{w(\ell)}\exp[-\rho^2/w(\ell)^2]
\exp\!\left[i\left(k\ell+\frac{k\rho^2\ell}{2(\ell^2+z_R^2)}-\arctan\frac{\ell}{z_R}\right)\right].$$

The signs of the positive carrier, curvature phase and negative Gouy phase follow $e^{-i\omega t}$. At the waist centre the full on-axis phase gradient is $(k-1/z_R)\hat{\mathbf l}$, because Gouy phase adds a small longitudinal correction. Its direction matches the prescribed wavevector; its magnitude is not exactly $k$. The carrier gradient is exactly $k\hat{\mathbf l}$. This distinction is part of the paraxial model, not a coordinate-rotation error.

- The map stores 81 depths and 1,601 detector-line samples. Geometry or order changes rebuild the analytical fields; phase changes reuse them; moving the detector reuses both fields and intensity maps.
- The main view remains a live Plotly figure; the linked lesson guides and optional xz panel use PNG images refreshed by the explorer. Reset restores the default pair and controls.
- Rerunning the experiment cell closes its previous explorer and reloads the visualization modules, avoiding stale classes after notebook updates; `gaussian_orders.close()` releases its widgets and arrays. No FFT, spectral window, or numerical boundary propagation is involved.

Tests compare detector-line quadrature with finer and wider grids, verify the local broad-beam plane-wave limit, and check widget reset and repeated execution. Those checks certify specified examples within this assigned model. They do not establish equivalence to the numerical mask experiment or to physical grating scattering.

The 3D shells use the fields' rotated coordinates and width law. They are clipped display contours, not apertures. The XY detector samples the same three-dimensional fields at 0.1 µm spacing in x and 0.4 µm in y. Its grid follows the square crop. The 3D detector averages display cells to reduce clutter; the adjacent map retains finer samples.

The green incident contour uses the same Gaussian waist scale, with propagation along negative $z$ and its waist at the grating origin. Its displayed segment lies above the grating. The outgoing unit coefficients remain independently assigned: the illustration does not impose a diffraction-efficiency or power-balance relation. Controls are grouped into beam/grating and orders/detector settings; depth means $s=-z_D$.

The detector uses a dark sensor background and a warm false-colour intensity palette, with the same fixed 0–4 intensity scale in the XY map and on the 3D detector. This is a signal display, not the visible colour of the light: the default 1 µm wavelength is infrared. No camera spectral response, digital counts, noise or saturation is modeled.

The detector and grating reference patches are square. Their display bounds include two estimated beam radii on either side of both axes, plus a 10% margin. The 3D plotting box remains cubic with equal physical scales and contains the incident segment and the path to the selected detector. Only the displayed window changes; the full-line overlap calculation is unchanged. The fitted crop follows depth and geometry but leaves the camera orientation intact.

- The main lesson defines nominal and local fringe spacing and works through the default example. For its local estimate, the implementation unwraps the relative phase along the detector line and differentiates it on the existing spatial samples.
- Values are displayed only where envelope intensity exceeds 1% of its line peak, the cross-term amplitude exceeds 0.1% of that peak, and amplitude-balance visibility exceeds 0.1. These are display masks, not a detector noise model or coherence criterion.
- When overlap is negligible, the rulers are hidden rather than presenting a fringe measurement in an empty tail.

The comparison pattern retains the same two envelopes but replaces relative phase by a linear carrier matched at the strongest-overlap point. This isolates phase-curvature effects without changing envelope amplitudes. The purple ruler reports a local phase-gradient length rather than measuring neighboring intensity maxima.

:::

### Index and Gaussian spreading

The assigned-beam formulas use the surrounding-medium wavelength:

$$z_R=\frac{\pi w_0^2}{\lambda}=\frac{\pi n w_0^2}{\lambda_0},\qquad
\theta_{\rm div}\simeq\frac{\lambda_0}{\pi n w_0}.$$

- At fixed vacuum wavelength and physical waist, increasing a real surrounding index increases the Rayleigh range and reduces paraxial divergence.
- To represent that change here, enter the new **medium wavelength**; there is no separate index control.
- Changing the physical grating's internal material index would instead require a scattering model and would not be represented merely by assigning a new Gaussian waist. 